# TSR-X growth ablations: resnet18/cifar100, seeds 42

Two jobs at a time, one per T4. Settings: **GPU T4 x2**, **Internet on**, run as a
committed version (Save Version -> Save & Run All). Outputs land in
`/kaggle/working/ablations`; copy `ablations/<cell>/*` into the local `results/ablations/<cell>/`.
If the repository is private, add a Kaggle secret `GITHUB_TOKEN` (read access).

In [ ]:
import os, subprocess
repo, branch = "https://github.com/Dutta-06/Topological-Self-Regulation.git", "rebuttal-ablations"
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    repo = repo.replace("https://", f"https://{tok}@")
except Exception:
    pass  # public repository, or no secret set
if not os.path.exists("/kaggle/working/tsr"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", branch, repo, "/kaggle/working/tsr"], check=True)
os.chdir("/kaggle/working/tsr")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
import torch
print(torch.__version__, torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 2, "select the GPU T4 x2 accelerator"

In [ ]:
# Smoke test first: a 500-step discovery run must finish cleanly before the slice starts.
!cd /kaggle/working/tsr && python -m bench.train_tsrx --arch resnet18 --dataset cifar100 --smoke-test --epochs 1 \
    --data-root /kaggle/working/data --num-workers 2 --out /kaggle/working/smoke/r18.pt | tail -6

In [ ]:
!cd /kaggle/working/tsr && python scripts/run_ablations.py --cells resnet18/cifar100 --seeds 42 --arms off random cost \
    --gpus 0 1 --num-workers 2 --stop-starting-after 8.5 \
    --data-root /kaggle/working/data --out-root /kaggle/working/ablations

In [ ]:
!cd /kaggle/working/tsr && python scripts/eval_ablations.py --root /kaggle/working/ablations --metric final
!rm -rf /kaggle/working/data /kaggle/working/smoke /kaggle/working/tsr   # keep the version output small
!du -sh /kaggle/working/ablations